# AEMO Evaluation Notebook

This notebook mirrors the evaluation workflow in `test_eval.ipynb`, but it is wired to the AEMO trajectories exported by `aemo_simrun.ipynb`. It loads the AEMO manifest, rehydrates the episode logs, and runs `evaluate_experiments()` on the resulting `all_logs` dictionary.

Use this notebook to inspect AEMO return statistics, recommended RTG / return scale values, and risk-sensitive summary metrics without touching the training notebook.

In [ ]:
from pathlib import Path
import json

import polars as pl

from aemo_notebook_utils import load_episode_logs_from_parquet
from helper import evaluate_experiments, bootstrap_confidence_intervals

REPO_ROOT = next(
    (candidate for candidate in [Path.cwd(), Path.cwd().parent] if (candidate / 'README.md').exists() and (candidate / 'src').exists()),
    Path.cwd(),
)

MANIFEST_PATH = REPO_ROOT / 'data' / 'aemo_dt' / 'aemo_dt_manifest.json'
EVAL_OUTPUT_DIR = REPO_ROOT / 'eval_output' / 'aemo_dt_compare'
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repo root: {REPO_ROOT}')
print(f'Manifest: {MANIFEST_PATH}')
print(f'Eval output: {EVAL_OUTPUT_DIR}')

In [ ]:
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f'Could not find AEMO manifest at {MANIFEST_PATH}. Run aemo_simrun.ipynb first.'
    )

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
raw_outputs = manifest.get('raw_outputs', {})

all_logs: dict[str, list[pl.DataFrame]] = {}
skipped_outputs: list[str] = []

for label, output_path in raw_outputs.items():
    if label.endswith('__incidents'):
        continue
    parquet_path = Path(output_path)
    if not parquet_path.is_file():
        skipped_outputs.append(label)
        print(f'Skipping missing parquet for {label}: {parquet_path}')
        continue
    episodes = load_episode_logs_from_parquet(parquet_path)
    if episodes:
        all_logs[label] = episodes

print(f'Loaded {len(all_logs)} AEMO experiment groups')
print('Labels:')
for label, episodes in sorted(all_logs.items()):
    print(f'  {label}: {len(episodes)} episodes')

if skipped_outputs:
    print('Missing outputs:')
    for label in skipped_outputs:
        print(f'  {label}')

In [ ]:
# Evaluate all AEMO trajectory groups and save the metrics table.
metrics = evaluate_experiments(all_logs, target_return=0.0, save_dir=str(EVAL_OUTPUT_DIR))
metrics_csv_path = EVAL_OUTPUT_DIR / 'evaluation_metrics.csv'
metrics.write_csv(str(metrics_csv_path))

print(f'Saved evaluation metrics to: {metrics_csv_path}')
metrics

In [ ]:
# Recommended RTG / return-scale summary.
best_overall_rtg = float(metrics['recommended_rtg'].max()) if 'recommended_rtg' in metrics.columns else 0.0
best_return_scale = float(metrics['recommended_return_scale'].max()) if 'recommended_return_scale' in metrics.columns else 1.0

print(f'Best recommended RTG across all AEMO runs: {best_overall_rtg}')
print(f'Best recommended return scale across all AEMO runs: {best_return_scale}')

summary_cols = [
    'experiment',
    'mean_reward',
    'std_reward',
    'recommended_rtg',
    'recommended_return_scale',
    'var_5',
    'cvar_5',
]
existing_cols = [col for col in summary_cols if col in metrics.columns]
print(metrics.select(existing_cols))

In [ ]:
# Bootstrap confidence intervals on the AEMO trajectories.
cis = bootstrap_confidence_intervals(
    all_logs,
    n_bootstrap=1000,
    confidence_level=0.95,
    seed=42,
)

for label, ci in sorted(cis.items()):
    print(
        f"{label:>40s}: mean={ci['mean']:+.2f}  95% CI=[{ci['ci_lower']:+.2f}, {ci['ci_upper']:+.2f}]  std={ci['std']:.2f}"
    )